# Data quality Assesment

This notebook contains the following list of data quality validation steps:

- Clean feature names
- Detection of missing values 
- Check cardinality
- Validate value ranges 
- Business logic consistency
- Optimize dataframe format for better management if needed

In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / 'data'
BRONZE_DIR = DATA_DIR / 'bronze'
DATA_FILE = BRONZE_DIR / 'PS_20174392719_1491204439457_log.csv'

In [2]:
df = pd.read_csv(DATA_FILE)
df

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.00,160296.36,M1979787155,0.00,0.00,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.00,19384.72,M2044282225,0.00,0.00,0,0
2,1,TRANSFER,181.00,C1305486145,181.00,0.00,C553264065,0.00,0.00,1,0
3,1,CASH_OUT,181.00,C840083671,181.00,0.00,C38997010,21182.00,0.00,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.00,29885.86,M1230701703,0.00,0.00,0,0
...,...,...,...,...,...,...,...,...,...,...,...
6362615,743,CASH_OUT,339682.13,C786484425,339682.13,0.00,C776919290,0.00,339682.13,1,0
6362616,743,TRANSFER,6311409.28,C1529008245,6311409.28,0.00,C1881841831,0.00,0.00,1,0
6362617,743,CASH_OUT,6311409.28,C1162922333,6311409.28,0.00,C1365125890,68488.84,6379898.11,1,0
6362618,743,TRANSFER,850002.52,C1685995037,850002.52,0.00,C2080388513,0.00,0.00,1,0


In [3]:
memory_usage = df.memory_usage(deep=True).sum()
print(f"Memory size: {memory_usage / (1024 ** 2):.2f} MB")

Memory size: 1598.19 MB


**Observations:**

The df is relatively big (~ 1.56 GB) and my computer is not the best so I will:
- use parquet instead of csv format to manage the dfs
- save plots as images 
- save heavy and slow functions inside the cache

## Clean feature names

CamelCase detected, let's change it to snake_case and lowercase only

In [4]:
data = df.copy()

rename_map = {
        'oldbalanceOrg': 'old_balance_orig',
        'newbalanceOrig': 'new_balance_orig',
        'oldbalanceDest': 'old_balance_dest',
        'newbalanceDest': 'new_balance_dest',
        'nameOrig': 'name_orig',
        'nameDest': 'name_dest',
        'isFraud': 'is_fraud',
        'isFlaggedFraud': 'is_flagged_fraud'
    }
    
rename_map = {k: v for k, v in rename_map.items() if k in data.columns}
data = data.rename(columns=rename_map)

if rename_map:
    print(f"Renamed {len(rename_map)} columns")
    for old, new in rename_map.items():
        print(f"{old} -> {new}")

else:
    print(" No columns to rename")

Renamed 8 columns
oldbalanceOrg -> old_balance_orig
newbalanceOrig -> new_balance_orig
oldbalanceDest -> old_balance_dest
newbalanceDest -> new_balance_dest
nameOrig -> name_orig
nameDest -> name_dest
isFraud -> is_fraud
isFlaggedFraud -> is_flagged_fraud


## Missing values

In [5]:
df.info(show_counts=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6362620 entries, 0 to 6362619
Data columns (total 11 columns):
 #   Column          Non-Null Count    Dtype  
---  ------          --------------    -----  
 0   step            6362620 non-null  int64  
 1   type            6362620 non-null  object 
 2   amount          6362620 non-null  float64
 3   nameOrig        6362620 non-null  object 
 4   oldbalanceOrg   6362620 non-null  float64
 5   newbalanceOrig  6362620 non-null  float64
 6   nameDest        6362620 non-null  object 
 7   oldbalanceDest  6362620 non-null  float64
 8   newbalanceDest  6362620 non-null  float64
 9   isFraud         6362620 non-null  int64  
 10  isFlaggedFraud  6362620 non-null  int64  
dtypes: float64(5), int64(3), object(3)
memory usage: 534.0+ MB


**Observations**: 
- 11 columns
- 6362620 rows
- No missing values!
- dtypes: 3 int64, 3 object, 5 float64

## Cardinality

In [6]:
for col in data.columns:
    unique_count = data[col].nunique()
    total_count = len(data)
    cardinality_ratio = unique_count / total_count
    
    if data[col].dtype in ['object']:
        var_type = 'Categorical'
        if unique_count <= 10:
            action = "Low" # these are good for encoding
        elif unique_count <= 50:
            action = "Medium" # grouping
        elif unique_count <= 1000:
            action = "High" # aggregate
        else:
            action = "Very High" # aggregate or drop
    else:
        var_type = 'Numerical'
        if cardinality_ratio < 0.01:
            action = "Low variance" # potentially ugly, ask yourself why
        else:
            action = "Normal" 
    
    print(f"{col:<20} {var_type:<12} {unique_count:>15,} {cardinality_ratio:>11.2%} {action:<25}")


step                 Numerical                743       0.01% Low variance             
type                 Categorical                5       0.00% Low                      
amount               Numerical          5,316,900      83.56% Normal                   
name_orig            Categorical        6,353,307      99.85% Very High                
old_balance_orig     Numerical          1,845,844      29.01% Normal                   
new_balance_orig     Numerical          2,682,586      42.16% Normal                   
name_dest            Categorical        2,722,362      42.79% Very High                
old_balance_dest     Numerical          3,614,697      56.81% Normal                   
new_balance_dest     Numerical          3,555,499      55.88% Normal                   
is_fraud             Numerical                  2       0.00% Low variance             
is_flagged_fraud     Numerical                  2       0.00% Low variance             


**Obervations:**

- step: Time step in hours of the month (1-743). Create new derived features later like step_day, step_hour, is_weekend or is_night
- type: Transaction type. Keep OHT or ordinal encoding
- amount: Continuous numeric feature. Maybe scale but let's see the distribution first
- name_orig: Extremely high, probably a unique identifier for each sender (account ID). Drop or use for aggregation (maybe prefix of origin or frequency). Also maybe hash encoding? Check on bivar
- old_balance_orig, new_balance_orig, old_balance_dest, new_balance_dest: Continuous numeric. Keep to check the balances later. Maybe they show some weird behaviour?
- name_dest: High-cardinality, same procedure as name_orig

- is_fraud: target
- is_flagged_fraud: target too

## Value ranges

In [7]:
neg_vals = []
for col in df.select_dtypes(include=[np.number]).columns:
    if (df[col] < 0).any():
        n_negative = (df[col] < 0).sum()
        neg_vals.append(f" This {col} has {n_negative:,} negative values")

if neg_vals:
    for val in neg_vals:
        print(neg_vals)
else:
    print("No obvious negative values detected")
    print("Great!")

No obvious negative values detected
Great!


## Business logic consistency

Here I'm going to check different assumptions that came to my mind:

### Assumption 1: for origin and destination accounts, balance arithmetic should be consistent, with this I mean that:

``` old_balance_orig - amount = new_balance_orig ```

&

``` old_balance_dest + amount = new_balance_dest```




In [9]:
balances_subset = data[['step', 
           'type', 
           'amount', 
           'old_balance_orig', 
           'new_balance_orig', 
           'old_balance_dest', 
           'new_balance_dest', 
           'is_fraud']].copy()

balances_subset['expected_new_balance_orig'] = balances_subset['old_balance_orig'] - balances_subset['amount']
balances_subset['balance_diff_orig'] = abs(balances_subset['new_balance_orig'] - balances_subset['expected_new_balance_orig'])


balances_subset['expected_new_balance_dest'] = balances_subset['old_balance_dest'] + balances_subset['amount']
balances_subset['balance_diff_dest'] = abs(balances_subset['new_balance_dest'] - balances_subset['expected_new_balance_dest'])
balances_subset

,step,type,amount,old_balance_orig,new_balance_orig,old_balance_dest,new_balance_dest,is_fraud,expected_new_balance_orig,balance_diff_orig,expected_new_balance_dest,balance_diff_dest
0,1,PAYMENT,9839.64,170136.00,160296.36,0.00,0.00,0,160296.36,0.0,9839.64,9.839640e+03
1,1,PAYMENT,1864.28,21249.00,19384.72,0.00,0.00,0,19384.72,0.0,1864.28,1.864280e+03
2,1,TRANSFER,181.00,181.00,0.00,0.00,0.00,1,0.00,0.0,181.00,1.810000e+02
3,1,CASH_OUT,181.00,181.00,0.00,21182.00,0.00,1,0.00,0.0,21363.00,2.136300e+04
4,1,PAYMENT,11668.14,41554.00,29885.86,0.00,0.00,0,29885.86,0.0,11668.14,1.166814e+04
...,...,...,...,...,...,...,...,...,...,...,...,...
6362615,743,CASH_OUT,339682.13,339682.13,0.00,0.00,339682.13,1,0.00,0.0,339682.13,0.000000e+00
6362616,743,TRANSFER,6311409.28,6311409.28,0.00,0.00,0.00,1,0.00,0.0,6311409.28,6.311409e+06
6362617,743,CASH_OUT,6311409.28,6311409.28,0.00,68488.84,6379898.11,1,0.00,0.0,6379898.12,1.000000e-02
6362618,743,TRANSFER,850002.52,850002.52,0.00,0.00,0.00,1,0.00,0.0,850002.52,8.500025e+05


Considering a tolerance of 0.01

In [ ]:
tolerance = 0.01

inconsistent_orig = balances_subset[balances_subset['balance_diff_orig'] > tolerance]
inconsistent_dest = balances_subset[balances_subset['balance_diff_dest'] > tolerance]

print(f"Total transactions: {len(data):,}")
print(f"Inconsistent origin balances: {len(inconsistent_orig):,} ({len(inconsistent_orig)/len(data)*100:.2f}%)")
print(f"Inconsistent origin balances: {len(inconsistent_dest):,} ({len(inconsistent_dest)/len(data)*100:.2f}%)")

Total transactions: 6,362,620
Inconsistent origin balances: 5,077,691 (79.81%)
Inconsistent origin balances: 4,188,647 (65.83%)


### Assumption 2: Zero amount transactions can be point out fraud

In [56]:
zero_amount = data.query("amount <= 0")
zero_amount

,step,type,amount,name_orig,old_balance_orig,new_balance_orig,name_dest,old_balance_dest,new_balance_dest,is_fraud,is_flagged_fraud
2736447,212,CASH_OUT,0.0,C1510987794,0.0,0.0,C1696624817,0.00,0.00,1,0
3247298,250,CASH_OUT,0.0,C521393327,0.0,0.0,C480398193,0.00,0.00,1,0
3760289,279,CASH_OUT,0.0,C539112012,0.0,0.0,C1106468520,538547.63,538547.63,1,0
5563714,387,CASH_OUT,0.0,C1294472700,0.0,0.0,C1325541393,7970766.57,7970766.57,1,0
5996408,425,CASH_OUT,0.0,C832555372,0.0,0.0,C1462759334,76759.90,76759.90,1,0
5996410,425,CASH_OUT,0.0,C69493310,0.0,0.0,C719711728,2921531.34,2921531.34,1,0
6168500,554,CASH_OUT,0.0,C10965156,0.0,0.0,C1493336195,230289.66,230289.66,1,0
6205440,586,CASH_OUT,0.0,C1303719003,0.0,0.0,C900608348,1328472.86,1328472.86,1,0
6266414,617,CASH_OUT,0.0,C1971175979,0.0,0.0,C1352345416,0.00,0.00,1,0
6281483,646,CASH_OUT,0.0,C2060908932,0.0,0.0,C1587892888,0.00,0.00,1,0


### Assumption 3: Round amounts can indicate fraud transactions

In [57]:
round_transactions = data.query("amount % 1 == 0")
round_transactions

,step,type,amount,name_orig,old_balance_orig,new_balance_orig,name_dest,old_balance_dest,new_balance_dest,is_fraud,is_flagged_fraud
2,1,TRANSFER,181.0,C1305486145,181.00,0.00,C553264065,0.00,0.00,1,0
3,1,CASH_OUT,181.0,C840083671,181.00,0.00,C38997010,21182.00,0.00,1,0
251,1,TRANSFER,2806.0,C1420196421,2806.00,0.00,C972765878,0.00,0.00,1,0
252,1,CASH_OUT,2806.0,C2101527076,2806.00,0.00,C1007251739,26202.00,0.00,1,0
256,1,PAYMENT,365.0,C600958416,319.00,0.00,M1884231057,0.00,0.00,0,0
...,...,...,...,...,...,...,...,...,...,...,...
6362580,741,TRANSFER,10000000.0,C88849251,25674547.89,15674547.89,C1939028448,0.00,0.00,1,0
6362581,741,CASH_OUT,10000000.0,C677394894,10000000.00,0.00,C1866259073,0.00,10000000.00,1,0
6362582,741,TRANSFER,10000000.0,C1945606464,15674547.89,5674547.89,C625944676,0.00,0.00,1,0
6362583,741,CASH_OUT,10000000.0,C1668034607,10000000.00,0.00,C1250722530,192912.98,10192912.98,1,0


In [60]:
rounded_10 = data.query("amount % 1 == 0")
rounded_10


,step,type,amount,name_orig,old_balance_orig,new_balance_orig,name_dest,old_balance_dest,new_balance_dest,is_fraud,is_flagged_fraud
2,1,TRANSFER,181.0,C1305486145,181.00,0.00,C553264065,0.00,0.00,1,0
3,1,CASH_OUT,181.0,C840083671,181.00,0.00,C38997010,21182.00,0.00,1,0
251,1,TRANSFER,2806.0,C1420196421,2806.00,0.00,C972765878,0.00,0.00,1,0
252,1,CASH_OUT,2806.0,C2101527076,2806.00,0.00,C1007251739,26202.00,0.00,1,0
256,1,PAYMENT,365.0,C600958416,319.00,0.00,M1884231057,0.00,0.00,0,0
...,...,...,...,...,...,...,...,...,...,...,...
6362580,741,TRANSFER,10000000.0,C88849251,25674547.89,15674547.89,C1939028448,0.00,0.00,1,0
6362581,741,CASH_OUT,10000000.0,C677394894,10000000.00,0.00,C1866259073,0.00,10000000.00,1,0
6362582,741,TRANSFER,10000000.0,C1945606464,15674547.89,5674547.89,C625944676,0.00,0.00,1,0
6362583,741,CASH_OUT,10000000.0,C1668034607,10000000.00,0.00,C1250722530,192912.98,10192912.98,1,0


In [59]:
fraud_round_transactions = round_transactions.query("is_fraud == 1")
fraud_round_transactions

,step,type,amount,name_orig,old_balance_orig,new_balance_orig,name_dest,old_balance_dest,new_balance_dest,is_fraud,is_flagged_fraud
2,1,TRANSFER,181.0,C1305486145,181.00,0.00,C553264065,0.00,0.00,1,0
3,1,CASH_OUT,181.0,C840083671,181.00,0.00,C38997010,21182.00,0.00,1,0
251,1,TRANSFER,2806.0,C1420196421,2806.00,0.00,C972765878,0.00,0.00,1,0
252,1,CASH_OUT,2806.0,C2101527076,2806.00,0.00,C1007251739,26202.00,0.00,1,0
680,1,TRANSFER,20128.0,C137533655,20128.00,0.00,C1848415041,0.00,0.00,1,0
...,...,...,...,...,...,...,...,...,...,...,...
6362580,741,TRANSFER,10000000.0,C88849251,25674547.89,15674547.89,C1939028448,0.00,0.00,1,0
6362581,741,CASH_OUT,10000000.0,C677394894,10000000.00,0.00,C1866259073,0.00,10000000.00,1,0
6362582,741,TRANSFER,10000000.0,C1945606464,15674547.89,5674547.89,C625944676,0.00,0.00,1,0
6362583,741,CASH_OUT,10000000.0,C1668034607,10000000.00,0.00,C1250722530,192912.98,10192912.98,1,0


TO-DO:

1. Do research on why there are inconsistencies in the transactions, obtain more information about how different types work.
2. Analyze the fraudulent transactions with amount = 0 or negative to find out why they happened and which patterns they follow.
3. 

In [17]:
data_temp = data.copy()
data_temp['expected_new_balance_orig'] = data_temp['old_balance_orig'] - data_temp['amount']
data_temp['balance_diff'] = abs(data_temp['new_balance_orig'] - data_temp['expected_new_balance_orig'])

# Check for inconsistencies (allow small floating point errors)
tolerance = 0.01
inconsistent = data_temp[data_temp['balance_diff'] > tolerance]

print(f"Total transactions: {len(data):,}")
print(f"Inconsistent origin balances: {len(inconsistent):,} ({len(inconsistent)/len(data)*100:.2f}%)")

if len(inconsistent) > 0:
    print(f"\n⚠️  ALERT: {len(inconsistent):,} transactions have balance arithmetic errors!")
    print("\nSample of inconsistencies:")
    print(inconsistent[['type', 'amount', 'old_balance_orig', 'new_balance_orig', 
                        'expected_new_balance_orig', 'balance_diff', 'is_fraud']].head(10))
    print("\n🔍 Possible reasons:")
    print("  - Transaction type might affect calculation (CASH_IN adds, CASH_OUT subtracts)")
    print("  - Fees or commissions not recorded in amount")
    print("  - Data quality issues")
    print("  - Fraudulent manipulation")
else:
    print("✅ All balance calculations are consistent!")

# Rule 2: Check by transaction type
print("\n\n2️⃣  Balance Consistency by Transaction Type")
print("-" * 90)
for txn_type in data['type'].unique():
    type_data = data_temp[data_temp['type'] == txn_type]
    inconsistent_type = type_data[type_data['balance_diff'] > tolerance]
    print(f"{txn_type:<15} Inconsistent: {len(inconsistent_type):>8,} / {len(type_data):>10,} "
          f"({len(inconsistent_type)/len(type_data)*100:>5.2f}%)")

Total transactions: 6,362,620
Inconsistent origin balances: 5,077,691 (79.81%)

⚠️  ALERT: 5,077,691 transactions have balance arithmetic errors!

Sample of inconsistencies:
        type     amount  old_balance_orig  new_balance_orig  \
8    PAYMENT    4024.36           2671.00               0.0   
10     DEBIT    9644.94           4465.00               0.0   
13   PAYMENT   11633.76          10127.00               0.0   
15  CASH_OUT  229133.94          15325.00               0.0   
16   PAYMENT    1563.82            450.00               0.0   
19  TRANSFER  215310.30            705.00               0.0   
24  TRANSFER  311685.89          10835.00               0.0   
25   PAYMENT    6061.13            443.00               0.0   
28   PAYMENT    8901.99           2958.91               0.0   
29   PAYMENT    9920.52              0.00               0.0   

    expected_new_balance_orig  balance_diff  is_fraud  
8                    -1353.36       1353.36         0  
10                  

In [18]:

# Rule 3: Zero amount with balance changes
print("\n\n3️⃣  Zero Amount Transactions with Balance Changes")
print("-" * 90)
zero_amount = data[(data['amount'] == 0) & 
                   ((data['old_balance_orig'] != data['new_balance_orig']) | 
                    (data['old_balance_dest'] != data['new_balance_dest']))]
print(f"Transactions with $0 amount but balance changes: {len(zero_amount):,}")
if len(zero_amount) > 0:
    print("⚠️  This is suspicious - investigate further!")
    print(zero_amount[['type', 'amount', 'oldbalanceOrig', 'newbalanceOrig', 'isFraud']].head())




3️⃣  Zero Amount Transactions with Balance Changes
------------------------------------------------------------------------------------------
Transactions with $0 amount but balance changes: 0



- Clean feature names
- Detection of missing values 
- Check cardinality
- Validate value ranges 
- Business logic consistency
- Optimize dataframe format for better management if needed